# Задание 5. Рекомендательная система Content-Based

## Постановка задачи

Необходимо разработать рекомендательную систему на основе метода Content-Based для датасета Goodbooks-10k Extended.

В отличие от методов User-Based и Item-Based KNN и SVD, Content-Based использует содержательные характеристики самих книг. Для каждой книги формируется текстовое представление на основе названия, авторов, жанров и описания. Затем для каждого пользователя строится профиль интересов по книгам, которые получили высокие оценки в обучающей выборке.

После этого книги ранжируются по сходству их содержания с профилем пользователя.

Для оценки качества используются Precision@K, Recall@K, NDCG@K и HitRate@K при K = 5, 10 и 20.

Основные гиперпараметры эксперимента:

* `MIN_RATING_LIKE` — минимальная оценка книги, используемая при построении профиля пользователя;
* `MAX_PROFILE_ITEMS` — максимальное число книг, используемых в профиле;
* `MAX_FEATURES` — максимальное число TF-IDF признаков;
* `NGRAM_RANGE` — диапазон используемых n-грамм.


## Структура и анализ данных

Используется Goodbooks-10k Extended.

Основным источником взаимодействий является `ratings.csv`, содержащий `user_id`, `book_id` и `rating`.

Файл `books_enriched.csv` содержит признаки книг. Для Content-Based используются `title`, `authors`, `genres` и `description`. Эти признаки описывают содержание объектов и позволяют вычислять сходство между книгами без непосредственного сравнения пользователей.

Для оценки используется то же случайное разбиение данных, что и в предыдущих экспериментах: 80 процентов для обучения и 20 процентов для тестирования, `random_state = 42`.

Релевантной считается книга с тестовой оценкой не ниже 4.

Таким образом, Content-Based использует информацию о содержании книг для построения рекомендаций, а тестовые оценки используются только для проверки качества.


In [2]:
import time, math, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import linear_kernel
warnings.filterwarnings('ignore')

DATA_DIR = 'goodbooks-10k-extended'
GLOBAL_PARAMS = {
    'TEST_SIZE': 0.20,
    'RANDOM_STATE': 42,
    'TOP_K_VALUES': [5, 10, 20],
    'RELEVANCE_THRESHOLD': 4,
    'N_EVAL_USERS': 200,
    'MIN_RATING_LIKE': 3,
    'MAX_PROFILE_ITEMS': 50,
    'MAX_FEATURES': 30000,
    'NGRAM_RANGE': (1, 2)
}

## Загрузка и анализ данных

Сначала загружаются оценки и метаданные книг. Проверяются размер датасета, средняя оценка, разреженность матрицы взаимодействий и наличие текстовых признаков.


In [3]:
def load_data():
    ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv', usecols=['user_id','book_id','rating'])
    books = pd.read_csv(f'{DATA_DIR}/books_enriched.csv')
    if 'book_id' not in books.columns: books = books.rename(columns={books.columns[0]:'book_id'})
    ratings = ratings.dropna(subset=['user_id','book_id','rating']).copy()
    books = books.dropna(subset=['book_id']).copy()
    ratings[['user_id','book_id']] = ratings[['user_id','book_id']].astype(np.int32)
    ratings['rating'] = ratings['rating'].astype(np.float32)
    books['book_id'] = books['book_id'].astype(np.int32)
    for col in ['title','authors','genres','description']:
        if col not in books.columns: books[col] = ''
        books[col] = books[col].fillna('').astype(str)
    return ratings, books

ratings_df, books_df = load_data()

n_users = ratings_df['user_id'].nunique()
n_books = ratings_df['book_id'].nunique()
n_ratings = len(ratings_df)
sparsity = (1 - n_ratings / (n_users * n_books)) * 100

print('=' * 60)
print('АНАЛИЗ GOODBOOKS-10K EXTENDED')
print('=' * 60)
print(f'Уникальных пользователей: {n_users:,}')
print(f'Уникальных книг:          {n_books:,}')
print(f'Всего оценок:             {n_ratings:,}')
print(f'Разреженность матрицы:    {sparsity:.2f}%')
print(f'Средняя оценка:           {ratings_df["rating"].mean():.3f}')
print('=' * 60)

display(books_df[['title','authors','genres','description']].head())
display(books_df[['title','authors','genres','description']].notna().sum().rename('count').to_frame())

АНАЛИЗ GOODBOOKS-10K EXTENDED
Уникальных пользователей: 53,424
Уникальных книг:          10,000
Всего оценок:             5,976,479
Разреженность матрицы:    98.88%
Средняя оценка:           3.920


,title,authors,genres,description
0,"The Hunger Games (The Hunger Games, #1)",['Suzanne Collins'],"['young-adult', 'fiction', 'fantasy', 'science...",WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...
1,Harry Potter and the Sorcerer's Stone (Harry P...,"['J.K. Rowling', 'Mary GrandPré']","['fantasy', 'fiction', 'young-adult', 'classics']",Harry Potter's life is miserable. His parents ...
2,"Twilight (Twilight, #1)",['Stephenie Meyer'],"['young-adult', 'fantasy', 'romance', 'fiction...",About three things I was absolutely positive.\...
3,To Kill a Mockingbird,['Harper Lee'],"['classics', 'fiction', 'historical-fiction', ...",The unforgettable novel of a childhood in a sl...
4,The Great Gatsby,['F. Scott Fitzgerald'],"['classics', 'fiction', 'historical-fiction', ...",Alternate Cover Edition ISBN: 0743273567 (ISBN...


,count
title,10000
authors,10000
genres,10000
description,10000


## Подготовка обучающей и тестовой выборок

Используется случайное разбиение 80 процентов на обучение и 20 процентов на тестирование с `random_state = 42`.

Для построения профиля пользователя используются только оценки из обучающей выборки. Книги с оценкой не ниже `MIN_RATING_LIKE` считаются положительно оценёнными.

Для ускорения экспериментов качество оценивается на случайной выборке из 200 пользователей, имеющих хотя бы одну релевантную книгу в тестовой выборке.


In [4]:
train_df, test_df = train_test_split(ratings_df, test_size=GLOBAL_PARAMS['TEST_SIZE'], random_state=GLOBAL_PARAMS['RANDOM_STATE'])

book_ids = pd.Index(np.sort(train_df['book_id'].unique()))
book_to_idx = pd.Series(np.arange(len(book_ids)), index=book_ids)

np.random.seed(GLOBAL_PARAMS['RANDOM_STATE'])
test_relevant = {uid:set(g.loc[g['rating'] >= GLOBAL_PARAMS['RELEVANCE_THRESHOLD'],'book_id']) for uid,g in test_df.groupby('user_id')}
candidate_users = np.array([uid for uid in train_df['user_id'].unique() if uid in test_relevant and len(test_relevant[uid]) > 0])
sample_eval_users = np.random.choice(candidate_users, size=min(GLOBAL_PARAMS['N_EVAL_USERS'],len(candidate_users)), replace=False)

print(f'Обучающая выборка: {len(train_df):,}')
print(f'Тестовая выборка:   {len(test_df):,}')
print(f'Книг в обучении:    {len(book_ids):,}')
print(f'Пользователей для оценки: {len(sample_eval_users)}')

Обучающая выборка: 4,781,183
Тестовая выборка:   1,195,296
Книг в обучении:    10,000
Пользователей для оценки: 200


## Подход к реализации Content-Based

Каждой книге сопоставляется текст, состоящий из `title`, `authors`, `genres` и `description`.

Для этого текста строится TF-IDF представление. TF-IDF уменьшает влияние слов, которые часто встречаются во всех документах, и увеличивает влияние слов, характерных для отдельных книг.

Профиль пользователя строится как взвешенное среднее TF-IDF-векторов понравившихся книг. В качестве веса используется оценка пользователя.

Затем для каждой непросмотренной книги вычисляется cosine similarity между вектором книги и профилем пользователя.

Чем выше сходство, тем выше книга располагается в рекомендациях.


In [5]:
def build_texts(books):
    data = books.set_index('book_id').reindex(book_ids).fillna('')
    return data['title'] + ' ' + data['authors'] + ' ' + data['genres'].str.replace('|',' ',regex=False) + ' ' + data['description']

book_texts = build_texts(books_df)

def build_tfidf(max_features=30000, ngram_range=(1,2)):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=2, max_df=0.95,
                                 sublinear_tf=True, strip_accents='unicode')
    start = time.time()
    tfidf = vectorizer.fit_transform(book_texts)
    elapsed = time.time() - start
    print(f'TF-IDF построен за {elapsed:.1f} сек.; книг: {tfidf.shape[0]:,}; признаков: {tfidf.shape[1]:,}')
    return vectorizer, tfidf, elapsed

vectorizer, tfidf_matrix, tfidf_time = build_tfidf(GLOBAL_PARAMS['MAX_FEATURES'], GLOBAL_PARAMS['NGRAM_RANGE'])

TF-IDF построен за 3.7 сек.; книг: 10,000; признаков: 30,000


## Формирование пользовательского профиля и рекомендации

`MAX_PROFILE_ITEMS` ограничивает число книг, участвующих в построении профиля.

Используются книги с наибольшими оценками пользователя. Их TF-IDF-векторы объединяются во взвешенный профиль. Благодаря этому книги с более высокой оценкой сильнее влияют на итоговый профиль пользователя.

После построения профиля уже просмотренные книги исключаются из рекомендаций.


In [6]:
def build_user_profile(uid, tfidf_matrix, min_rating_like=3, max_profile_items=50):
    user_train = train_df[train_df['user_id'] == uid]
    liked = user_train[user_train['rating'] >= min_rating_like].sort_values('rating', ascending=False).head(max_profile_items)
    liked = liked[liked['book_id'].isin(book_to_idx.index)]
    if liked.empty: return None
    idxs = liked['book_id'].map(book_to_idx).to_numpy()
    weights = liked['rating'].to_numpy(dtype=np.float32)
    profile = tfidf_matrix[idxs].multiply(weights[:,None]).sum(axis=0)
    return normalize(csr_matrix(profile))

def recommend_user(uid, tfidf_matrix, min_rating_like=3, max_profile_items=50, top_n=20):
    profile = build_user_profile(uid, tfidf_matrix, min_rating_like, max_profile_items)
    if profile is None: return pd.DataFrame(columns=['book_id','score'])
    scores = linear_kernel(profile, tfidf_matrix).ravel()
    seen_ids = set(train_df.loc[train_df['user_id'] == uid, 'book_id'])
    seen_idx = [book_to_idx[x] for x in seen_ids if x in book_to_idx.index]
    if seen_idx: scores[seen_idx] = -np.inf
    top_idx = np.argsort(-scores)[:top_n]
    return pd.DataFrame({'book_id':book_ids[top_idx], 'score':scores[top_idx]})

## Оценка качества

Для каждого пользователя строится список рекомендаций, после чего он сравнивается с релевантными книгами тестовой выборки.

Рассчитываются четыре метрики:

* Precision@K;
* Recall@K;
* HitRate@K;
* NDCG@K.

Исследование проводится при K = 5, 10 и 20.

Во время длительных вычислений выводится количество обработанных пользователей и приблизительное оставшееся время.


In [7]:
def evaluate_content(tfidf_matrix, min_rating_like=3, max_profile_items=50, k_values=(5,10,20), progress_label=''):
    metrics_by_k = {k:[] for k in k_values}
    start = time.time()
    for i,uid in enumerate(sample_eval_users,1):
        recs = recommend_user(uid, tfidf_matrix, min_rating_like, max_profile_items, max(k_values))
        if recs.empty: continue
        relevant = test_relevant.get(uid,set())
        for k in k_values:
            recommended = recs.head(k)['book_id'].tolist()
            hits = [1 if x in relevant else 0 for x in recommended]
            num_hits = sum(hits)
            precision = num_hits / k
            recall = num_hits / len(relevant) if relevant else 0
            hit_rate = 1.0 if num_hits > 0 else 0.0
            dcg = sum(hit / math.log2(j+2) for j,hit in enumerate(hits))
            idcg = sum(1 / math.log2(j+2) for j in range(min(k,len(relevant))))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            metrics_by_k[k].append((precision,recall,hit_rate,ndcg))
        if i == 1 or i % 10 == 0 or i == len(sample_eval_users):
            elapsed = time.time() - start
            speed = i / elapsed if elapsed > 0 else 0
            eta = (len(sample_eval_users)-i) / speed if speed > 0 else 0
            print(f'{progress_label} пользователи: {i}/{len(sample_eval_users)} ({100*i/len(sample_eval_users):.1f}%), осталось: {eta:.1f} сек.',end='\r')
    print()
    result = {}
    for k,values in metrics_by_k.items():
        m = np.mean(values,axis=0) if values else np.zeros(4)
        result.update({f'Precision@{k}':m[0],f'Recall@{k}':m[1],f'HitRate@{k}':m[2],f'NDCG@{k}':m[3]})
    return result

## Исследование влияния MIN_RATING_LIKE и MAX_PROFILE_ITEMS

Сначала исследуется влияние параметров пользовательского профиля.

`MIN_RATING_LIKE` определяет минимальную оценку книги, при которой она считается положительным примером. Низкое значение позволяет использовать больше книг, но может добавить в профиль менее характерные интересы пользователя.

`MAX_PROFILE_ITEMS` ограничивает число книг, участвующих в формировании профиля. Маленькое значение делает профиль более узким, а увеличение этого параметра позволяет учитывать больше интересов пользователя.

В этой серии экспериментов используется `NGRAM_RANGE = (1, 2)`. После выбора конфигурации профиля отдельно исследуется влияние различных диапазонов n-грамм.


In [8]:
experiment_results = []
profile_grid = [(3,20),(3,50),(4,20),(4,50),(4,100),(5,20),(5,50)]

start_all = time.time()
for j,(min_like,max_profile) in enumerate(profile_grid,1):
    print(f'\nЭксперимент {j}/{len(profile_grid)}: MIN_RATING_LIKE={min_like}, MAX_PROFILE_ITEMS={max_profile}')
    metrics = evaluate_content(tfidf_matrix,min_like,max_profile,GLOBAL_PARAMS['TOP_K_VALUES'],f'[{j}/{len(profile_grid)}]')
    metrics['MIN_RATING_LIKE'] = min_like
    metrics['MAX_PROFILE_ITEMS'] = max_profile
    metrics['NGRAM_RANGE'] = '(1, 2)'
    experiment_results.append(metrics)

results_df = pd.DataFrame(experiment_results)
display(results_df.round(6))
print(f'Время экспериментов: {time.time()-start_all:.1f} сек.')

best_profile_idx = results_df['NDCG@10'].idxmax()
best_profile = results_df.loc[best_profile_idx]

print('\nЛучшая конфигурация профиля по NDCG@10:')
display(best_profile.to_frame().round(6))


Эксперимент 1/7: MIN_RATING_LIKE=3, MAX_PROFILE_ITEMS=20
[1/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 2/7: MIN_RATING_LIKE=3, MAX_PROFILE_ITEMS=50
[2/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 3/7: MIN_RATING_LIKE=4, MAX_PROFILE_ITEMS=20
[3/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 4/7: MIN_RATING_LIKE=4, MAX_PROFILE_ITEMS=50
[4/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 5/7: MIN_RATING_LIKE=4, MAX_PROFILE_ITEMS=100
[5/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 6/7: MIN_RATING_LIKE=5, MAX_PROFILE_ITEMS=20
[6/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

Эксперимент 7/7: MIN_RATING_LIKE=5, MAX_PROFILE_ITEMS=50
[7/7] пользователи: 200/200 (100.0%), осталось: 0.0 сек.


,Precision@5,Recall@5,HitRate@5,NDCG@5,Precision@10,Recall@10,HitRate@10,NDCG@10,Precision@20,Recall@20,HitRate@20,NDCG@20,MIN_RATING_LIKE,MAX_PROFILE_ITEMS,NGRAM_RANGE
0,0.119000,0.041433,0.375000,0.117596,0.100500,0.072159,0.500000,0.108425,0.076000,0.106741,0.615000,0.105492,3,20,"(1, 2)"
1,0.137000,0.049454,0.440000,0.140402,0.111000,0.076964,0.550000,0.124106,0.077250,0.102346,0.605000,0.113551,3,50,"(1, 2)"
2,0.130000,0.046733,0.405000,0.127336,0.102500,0.073605,0.495000,0.112570,0.075000,0.104969,0.625000,0.106788,4,20,"(1, 2)"
3,0.138000,0.051794,0.435000,0.137846,0.107500,0.079014,0.520000,0.121239,0.078500,0.108863,0.620000,0.114782,4,50,"(1, 2)"
4,0.140000,0.052473,0.440000,0.143707,0.107500,0.079536,0.530000,0.124090,0.075500,0.106074,0.620000,0.114496,4,100,"(1, 2)"
5,0.110553,0.037551,0.356784,0.110358,0.092965,0.061668,0.467337,0.100131,0.064070,0.083994,0.542714,0.090829,5,20,"(1, 2)"
6,0.128643,0.043083,0.422111,0.131420,0.102010,0.067512,0.527638,0.113760,0.074121,0.095452,0.603015,0.105497,5,50,"(1, 2)"


Время экспериментов: 40.5 сек.

Лучшая конфигурация профиля по NDCG@10:


,1
Precision@5,0.137
Recall@5,0.049454
HitRate@5,0.44
NDCG@5,0.140402
Precision@10,0.111
Recall@10,0.076964
HitRate@10,0.55
NDCG@10,0.124106
Precision@20,0.07725
Recall@20,0.102346


## Исследование влияния NGRAM_RANGE

После определения оптимальных параметров профиля отдельно исследуется влияние диапазона n-грамм.

Рассматриваются шесть вариантов:

* `NGRAM_RANGE = (1, 1)`
* `NGRAM_RANGE = (1, 2)`
* `NGRAM_RANGE = (1, 3)`
* `NGRAM_RANGE = (1, 4)`
* `NGRAM_RANGE = (1, 5)`
* `NGRAM_RANGE = (1, 6)`

Использование более длинных n-грамм позволяет учитывать устойчивые словосочетания и последовательности слов в названии и описании книг. Одновременно увеличивается количество признаков и вычислительная стоимость построения TF-IDF.

Все варианты сравниваются при одинаковых значениях `MIN_RATING_LIKE` и `MAX_PROFILE_ITEMS`, выбранных на предыдущем этапе. Поэтому различия между результатами данной серии связаны именно с изменением представления текста.


In [13]:
ngram_ranges = [(1,1),(1,2),(1,3),(1,4),(1,5),(1,6)]
ngram_results = []

best_min_like = int(best_profile['MIN_RATING_LIKE'])
best_profile_items = int(best_profile['MAX_PROFILE_ITEMS'])

for j,ngram in enumerate(ngram_ranges,1):
    print(f'\nTF-IDF вариант {j}/{len(ngram_ranges)}: NGRAM_RANGE={ngram}')
    vec_start = time.time()
    vec = TfidfVectorizer(max_features=GLOBAL_PARAMS['MAX_FEATURES'], ngram_range=ngram,
                          min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode')
    matrix = vec.fit_transform(book_texts)
    vec_time = time.time() - vec_start
    print(f'Матрица построена за {vec_time:.1f} сек., признаков: {matrix.shape[1]:,}')

    metrics = evaluate_content(matrix,best_min_like,best_profile_items,
                               GLOBAL_PARAMS['TOP_K_VALUES'],f'[ngram {ngram}]')
    metrics['NGRAM_RANGE'] = str(ngram)
    metrics['MAX_FEATURES'] = GLOBAL_PARAMS['MAX_FEATURES']
    metrics['N_FEATURES_REAL'] = matrix.shape[1]
    metrics['TFIDF_TIME'] = vec_time
    ngram_results.append(metrics)

ngram_df = pd.DataFrame(ngram_results)
display(ngram_df[['NGRAM_RANGE','N_FEATURES_REAL','TFIDF_TIME',
                  'Precision@5','Recall@5','NDCG@5','HitRate@5',
                  'Precision@10','Recall@10','NDCG@10','HitRate@10',
                  'Precision@20','Recall@20','NDCG@20','HitRate@20']].round(6))


TF-IDF вариант 1/6: NGRAM_RANGE=(1, 1)
Матрица построена за 1.2 сек., признаков: 30,000
[ngram (1, 1)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

TF-IDF вариант 2/6: NGRAM_RANGE=(1, 2)
Матрица построена за 3.8 сек., признаков: 30,000
[ngram (1, 2)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

TF-IDF вариант 3/6: NGRAM_RANGE=(1, 3)
Матрица построена за 8.6 сек., признаков: 30,000
[ngram (1, 3)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

TF-IDF вариант 4/6: NGRAM_RANGE=(1, 4)
Матрица построена за 13.8 сек., признаков: 30,000
[ngram (1, 4)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

TF-IDF вариант 5/6: NGRAM_RANGE=(1, 5)
Матрица построена за 19.7 сек., признаков: 30,000
[ngram (1, 5)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.

TF-IDF вариант 6/6: NGRAM_RANGE=(1, 6)
Матрица построена за 26.0 сек., признаков: 30,000
[ngram (1, 6)] пользователи: 200/200 (100.0%), осталось: 0.0 сек.


,NGRAM_RANGE,N_FEATURES_REAL,TFIDF_TIME,Precision@5,Recall@5,NDCG@5,HitRate@5,Precision@10,Recall@10,NDCG@10,HitRate@10,Precision@20,Recall@20,NDCG@20,HitRate@20
0,"(1, 1)",30000,1.179020,0.090,0.031048,0.090837,0.330,0.0720,0.048398,0.079965,0.405,0.05425,0.072940,0.077501,0.535
1,"(1, 2)",30000,3.767333,0.137,0.049454,0.140402,0.440,0.1110,0.076964,0.124106,0.550,0.07725,0.102346,0.113551,0.605
2,"(1, 3)",30000,8.637897,0.148,0.052476,0.150849,0.470,0.1140,0.078881,0.129650,0.570,0.08375,0.111002,0.121420,0.645
3,"(1, 4)",30000,13.794348,0.155,0.054854,0.159663,0.490,0.1185,0.081479,0.135700,0.580,0.08600,0.113251,0.126077,0.650
4,"(1, 5)",30000,19.675196,0.155,0.055387,0.162525,0.495,0.1180,0.081363,0.137445,0.580,0.08600,0.113550,0.127879,0.650
5,"(1, 6)",30000,26.010906,0.154,0.055740,0.160713,0.490,0.1165,0.079411,0.135340,0.580,0.08700,0.114148,0.127746,0.650


## Анализ результатов исследования NGRAM_RANGE

Основным критерием выбора является `NDCG@10`, поскольку эта метрика учитывает не только наличие релевантных книг, но и их положение в рекомендациях.

Дополнительно анализируются `Precision@10`, `Recall@10` и `HitRate@10`.

По таблице сравнивается влияние увеличения длины n-грамм на качество рекомендаций и вычислительную стоимость построения TF-IDF.

Если увеличение диапазона n-грамм приводит к росту NDCG@10, это означает, что для данной выборки полезна дополнительная информация о словосочетаниях и последовательностях слов.

Если же качество снижается при добавлении длинных n-грамм, это может означать, что более специфические текстовые конструкции становятся слишком редкими и ухудшают обобщение модели.


In [17]:
best_ngram_idx = ngram_df['NDCG@10'].idxmax()
best_ngram = ngram_df.loc[best_ngram_idx]

print('Лучший вариант NGRAM_RANGE по NDCG@10:')
display(best_ngram.to_frame().round(6))

best_base = best_profile.copy()

if best_ngram['NDCG@10'] > best_base['NDCG@10']:
    best_content = best_ngram.copy()
    best_content['MIN_RATING_LIKE'] = best_min_like
    best_content['MAX_PROFILE_ITEMS'] = best_profile_items
else:
    best_content = best_base.copy()

print('\nЛучшая конфигурация Content-Based:')
print(f'MIN_RATING_LIKE = {int(best_content["MIN_RATING_LIKE"])}')
print(f'MAX_PROFILE_ITEMS = {int(best_content["MAX_PROFILE_ITEMS"])}')
print(f'NGRAM_RANGE = {best_content["NGRAM_RANGE"]}')
print(f'Precision@10 = {best_content["Precision@10"]:.6f}')
print(f'Recall@10 = {best_content["Recall@10"]:.6f}')
print(f'NDCG@10 = {best_content["NDCG@10"]:.6f}')
print(f'HitRate@10 = {best_content["HitRate@10"]:.6f}')

Лучший вариант NGRAM_RANGE по NDCG@10:


,4
Precision@5,0.155
Recall@5,0.055387
HitRate@5,0.495
NDCG@5,0.162525
Precision@10,0.118
Recall@10,0.081363
HitRate@10,0.58
NDCG@10,0.137445
Precision@20,0.086
Recall@20,0.11355



Лучшая конфигурация Content-Based:
MIN_RATING_LIKE = 3
MAX_PROFILE_ITEMS = 50
NGRAM_RANGE = (1, 5)
Precision@10 = 0.118000
Recall@10 = 0.081363
NDCG@10 = 0.137445
HitRate@10 = 0.580000


## Демонстрация на конкретном пользователе

Для одного пользователя показываются книги с высокими оценками в обучающей выборке и рекомендации лучшей конфигурации Content-Based.

Это позволяет проверить, насколько рекомендованные книги связаны с интересами пользователя по названию, авторам, жанрам и описанию.


In [18]:
example_user = int(sample_eval_users[0])
best_ngram_tuple = eval(best_content['NGRAM_RANGE']) if isinstance(best_content['NGRAM_RANGE'],str) else best_content['NGRAM_RANGE']

best_vectorizer = TfidfVectorizer(max_features=GLOBAL_PARAMS['MAX_FEATURES'], ngram_range=best_ngram_tuple,
                                   min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode')
best_matrix = best_vectorizer.fit_transform(book_texts)

print(f'Пользователь: {example_user}')
print('\nКниги с высокими оценками:')
display(train_df[train_df['user_id'] == example_user].sort_values('rating',ascending=False).head(10))

recs = recommend_user(example_user,best_matrix,int(best_content['MIN_RATING_LIKE']),
                      int(best_content['MAX_PROFILE_ITEMS']),10)
rec_display = recs.merge(books_df[['book_id','title','authors','genres']],on='book_id',how='left')

print('\nРекомендации Content-Based:')
display(rec_display)

Пользователь: 34618

Книги с высокими оценками:


,user_id,book_id,rating
5756153,34618,6928,5.0
5365879,34618,772,5.0
5102987,34618,4332,5.0
3960749,34618,4684,5.0
5614228,34618,4503,5.0
5516559,34618,130,4.0
5518884,34618,58,4.0
3968499,34618,4330,4.0
5516561,34618,2709,4.0
4095770,34618,28,4.0



Рекомендации Content-Based:


,book_id,score,title,authors,genres
0,361,0.226923,Oliver Twist,"['Charles Dickens', 'George Cruikshank', 'Phil...","['classics', 'fiction', 'historical-fiction']"
1,3868,0.226059,The Symposium,"['Plato', 'Christopher Gill']","['philosophy', 'classics', 'nonfiction', 'hist..."
2,3949,0.221926,Politics,['Aristotle'],"['philosophy', 'classics', 'nonfiction', 'hist..."
3,590,0.218566,David Copperfield,"['Charles Dickens', 'Jeremy Tambling']","['classics', 'fiction', 'historical-fiction']"
4,9667,0.214050,The Gay Science,"['Friedrich Nietzsche', 'Walter Kaufmann']","['philosophy', 'nonfiction', 'classics', 'poet..."
5,366,0.212168,John Adams,['David McCullough'],"['history', 'biography', 'nonfiction']"
6,7997,0.211037,Washington Square,"['Henry James', 'Michael Cunningham']","['classics', 'fiction', 'historical-fiction', ..."
7,3642,0.208848,Moll Flanders,['Daniel Defoe'],"['classics', 'fiction', 'historical-fiction']"
8,4880,0.207918,Apology,"['Plato', 'James J. Helm']","['philosophy', 'classics', 'nonfiction', 'hist..."
9,9605,0.207662,Essays and Poems,"['[Ralph Waldo Emerson', 'Peter Norberg]']","['poetry', 'classics', 'philosophy', 'nonficti..."


## Сравнение с User-Based + Item-Based KNN и SVD

Для итогового сравнения используются результаты предыдущих экспериментов при `K = 10`.

Для Content-Based используются метрики лучшей конфигурации, найденной в текущем исследовании.

In [19]:
knn_metrics = pd.DataFrame({
    'Precision@10':[0.210606],
    'Recall@10':[0.144350],
    'NDCG@10':[0.259965],
    'HitRate@10':[0.833333]
},index=['User+Item KNN'])

svd_metrics = pd.DataFrame({
    'Precision@10':[0.106000],
    'Recall@10':[0.073089],
    'NDCG@10':[0.140460],
    'HitRate@10':[0.535000]
},index=['SVD (10)'])

content_metrics = pd.DataFrame({
    'Precision@10':[best_content['Precision@10']],
    'Recall@10':[best_content['Recall@10']],
    'NDCG@10':[best_content['NDCG@10']],
    'HitRate@10':[best_content['HitRate@10']]
},index=['Content-Based (best)'])

comparison_df = pd.concat([knn_metrics,svd_metrics,content_metrics])
display(comparison_df.round(6))

,Precision@10,Recall@10,NDCG@10,HitRate@10
User+Item KNN,0.210606,0.144350,0.259965,0.833333
SVD (10),0.106000,0.073089,0.140460,0.535000
Content-Based (best),0.118000,0.081363,0.137445,0.580000


## Интерпретация результатов

Лучшей конфигурацией Content-Based стала `MIN_RATING_LIKE = 3`, `MAX_PROFILE_ITEMS = 50` и `NGRAM_RANGE = (1, 5)`.

Исследование показало, что увеличение диапазона n-грамм в целом улучшает качество рекомендаций. Переход от отдельных слов к биграммам, триграммам и более длинным сочетаниям постепенно повышает качество, однако после `(1,5)` дальнейшее увеличение до `(1,6)` уже не даёт улучшения основных метрик.

При этом использование более длинных n-грамм заметно увеличивает время построения TF-IDF. Поэтому `(1,5)` можно считать наиболее удачным вариантом среди исследованных с точки зрения соотношения качества и вычислительной сложности.

По сравнению с предыдущими моделями Content-Based показывает более высокое качество, чем SVD(10), но уступает User+Item KNN. Это показывает, что содержательные признаки книг хорошо дополняют информацию о взаимодействиях пользователей, но сами по себе не полностью отражают пользовательские предпочтения.


## Выводы

В работе была разработана рекомендательная система Content-Based для Goodbooks-10k Extended. Для построения рекомендаций использовались название, авторы, жанры и описание книг, представленные с помощью TF-IDF.

Было исследовано влияние `MIN_RATING_LIKE`, `MAX_PROFILE_ITEMS` и различных диапазонов `NGRAM_RANGE` от `(1,1)` до `(1,6)`. Лучший результат был получен при `MIN_RATING_LIKE = 3`, `MAX_PROFILE_ITEMS = 50` и `NGRAM_RANGE = (1,5)`.

Эксперименты показали, что добавление более длинных n-грамм улучшает качество рекомендаций до определённого предела, после которого рост вычислительной сложности уже не сопровождается улучшением результата.

Content-Based уступает User+Item KNN, но превосходит SVD(10), поэтому его целесообразно использовать как отдельный подход и как компонент итоговой гибридной системы.
